# Preprocessing — krok 1: konsolidacja obserwacji

Cel: z wielu obserwacji tej samej pary (kurs, przystanek), zapisywanych co 15 s,
wybrać jedną, ostateczną wartość.
Praca nadal na próbce 4 dni (22, 23, 25, 26 października), feed A.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from config import RAW
from src.analysis.consolidate import consolidate_trip_updates

skonsolidowane = consolidate_trip_updates(RAW, feed="A")
print("Liczba wierszy po konsolidacji:", f"{len(skonsolidowane):,}")
skonsolidowane.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Liczba wierszy po konsolidacji: 686,871


,date,trip_id,stop_id,route_id,stop_sequence,arrival_delay,departure_delay,arrival_time,liczba_obserwacji,ostatnia_obserwacja,niepewny_kurs
0,2026-10-22,20261021_1_129496644_8,12886,610,NaN,0.0,0.0,1.792629e+09,86,1792628656,False
1,2026-10-22,20261021_1_129497639_13,11104,937,NaN,0.0,0.0,1.792628e+09,12,1792627405,False
2,2026-10-22,20261021_1_129496706_6,12961,669,NaN,0.0,0.0,1.792628e+09,31,1792627712,False
3,2026-10-22,20261021_1_129497589_7,13737,904,NaN,0.0,0.0,1.792629e+09,84,1792628624,False
4,2026-10-22,20261021_1_129497589_7,12851,904,NaN,0.0,0.0,1.792629e+09,124,1792629298,False


In [2]:
import duckdb
from src.analysis.io import trip_updates_glob, READ_PARQUET_OPTS

zapytanie = f"""
WITH pary AS (
    SELECT
        date, trip_id, stop_id,
        count(*) AS n_total,
        count(stop_sequence) AS n_z_sequence
    FROM read_parquet('{trip_updates_glob(RAW)}', {READ_PARQUET_OPTS})
    WHERE feed = 'A'
    GROUP BY date, trip_id, stop_id
)
SELECT
    count(*) AS wszystkie_pary,
    sum(CASE WHEN n_z_sequence > 0 AND n_z_sequence < n_total THEN 1 ELSE 0 END) AS pary_niespojne
FROM pary
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 wszystkie_pary  pary_niespojne
         686871          3034.0


In [3]:
zapytanie_kontrolne = f"""
SELECT count(*) AS n
FROM (
    SELECT DISTINCT date, trip_id, stop_id
    FROM read_parquet('{trip_updates_glob(RAW)}', {READ_PARQUET_OPTS})
    WHERE feed = 'A'
)
"""
liczba_unikalnych_par = duckdb.query(zapytanie_kontrolne).to_df()["n"].iloc[0]

assert len(skonsolidowane) == liczba_unikalnych_par, (
    f"Niezgodność: konsolidacja dała {len(skonsolidowane)} wierszy, "
    f"a unikalnych par jest {liczba_unikalnych_par}"
)
print(f"OK: {len(skonsolidowane):,} wierszy = {liczba_unikalnych_par:,} unikalnych par")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OK: 686,871 wierszy = 686,871 unikalnych par


## Test spójności i poprawka

Pierwsza wersja funkcji grupowała dane po `(date, trip_id, stop_id, stop_sequence)`.
Oparto to na założeniu, że `stop_sequence` jest spójne w obrębie pary (kurs, przystanek) we wszystkich
jej obserwacjach. **Założenie okazało się błędne** dla 3034 z 686871 par (~0.44%).
Pole bywa puste w części obserwacji tej samej pary, a wypełnione w innych.

Błąd skutkował tym, że takie pary były sztucznie dzielone na dwie grupy, każda z niepełnym
`arg_max`. Naniesiono poprawkę: `stop_sequence` zostało przeniesione
z klucza grupowania do agregacji, tak jak opóźnienia.

Po poprawce liczba wierszy po konsolidacji = liczba unikalnych par źródłowych
(686 871) - zweryfikowane asercją.

In [4]:
zapytanie = f"""
SELECT
    date,
    count(*) AS wszystkie_pary,
    sum(CASE WHEN ostatnia_obserwacja - arrival_time > 1800 THEN 1 ELSE 0 END) AS podejrzane_30min,
    sum(CASE WHEN ostatnia_obserwacja - arrival_time > 3600 THEN 1 ELSE 0 END) AS podejrzane_1h
FROM (
    SELECT
        date, trip_id, stop_id,
        arg_max(arrival_time, observed_at) AS arrival_time,
        max(observed_at) AS ostatnia_obserwacja
    FROM read_parquet('{trip_updates_glob(RAW)}', {READ_PARQUET_OPTS})
    WHERE feed = 'A'
    GROUP BY date, trip_id, stop_id
)
GROUP BY date
ORDER BY date
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      date  wszystkie_pary  podejrzane_30min  podejrzane_1h
2026-10-22          210881              16.0           10.0
2026-10-23          215670             163.0          159.0
2026-10-25          125285               5.0            0.0
2026-10-26          135035             315.0          314.0


In [5]:
zapytanie = f"""
WITH pary AS (
    SELECT
        trip_id, route_id, stop_id,
        arg_max(arrival_delay, observed_at) AS arrival_delay,
        arg_max(arrival_time, observed_at)  AS arrival_time,
        max(observed_at)                    AS ostatnia_obserwacja
    FROM read_parquet('{trip_updates_glob(RAW)}', {READ_PARQUET_OPTS})
    WHERE feed = 'A' AND date = '2026-10-23'
    GROUP BY trip_id, route_id, stop_id
)
SELECT trip_id, route_id, count(*) AS ile_przystankow,
       min(ostatnia_obserwacja - arrival_time) AS min_roznica,
       max(ostatnia_obserwacja - arrival_time) AS max_roznica,
       max(arrival_delay) AS max_opoznienie
FROM pary
WHERE ostatnia_obserwacja - arrival_time > 3600
GROUP BY trip_id, route_id
ORDER BY ile_przystankow DESC
LIMIT 15
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                trip_id route_id  ile_przystankow  min_roznica  max_roznica  max_opoznienie
 20261021_4_129500127_2      132               29      44616.0      47016.0             0.0
 20261021_1_129178418_2        1               27      62827.0      65227.0          2154.0
20261021_5_129514722_13      128               26      36097.0      38257.0             0.0
20261021_5_129514723_14      128               23      32497.0      34597.0             0.0
 20261021_4_129504803_3      184               17      62894.0      64334.0          3372.0
 20261021_4_129512419_2      708               15      47753.0      48953.0             0.0
 20261021_4_129504802_2      184               15      64394.0      65594.0          3852.0
 20261021_1_129178417_1        1                5      65287.0      65647.0          2514.0
 20261021_4_129511153_3      513                1       8881.0       8881.0             0.0
 20261021_4_129511152_2      513                1      17016.0      17016.0     

## Dodatkowy sygnał - "niepewny_kurs"

Zaobserwowano niewielką grupę par (niewielkie ułamki procenta danych), gdzie ostatnia
obserwacja nastąpiła znacząco po planowym czasie przyjazdu. Dwa rozpoznane warianty:

- **Realnie rosnące opóźnienie** (np. linia 146 26.10) - kurs prawdopodobnie odwołany/zakłócony,
  ale nieoznaczony jako taki w źródle.
- **Nieusunięte kursy-widma** (np. 23.10, route_id 184/513/708) - wpisy ze `service_date` sprzed 1-2 dni,
  z zamrożonym opóźnieniem 0, wciąż widoczne w strumieniu RT.

Oba mechanizmy łączy wspólny, mierzalny symptom (duża różnica czasowa między ostatnią obserwacją
a planowym przyjazdem), więc oznaczono je jedną flagą `niepewny_kurs` (próg 2h, spójny
z progiem przyjętym dla ekstremalnych opóźnień).

In [3]:
skonsolidowane = consolidate_trip_updates(RAW, feed="A")
print("Par z flaga uncertain_completion:", skonsolidowane["uncertain_completion"].sum())
print("To", round(100 * skonsolidowane["uncertain_completion"].mean(), 3), "% wszystkich par")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Par z flagą niepewny_kurs: 482
To 0.073 % wszystkich par
